# UK Hansard Cryptoasset Regulation — Final Analysis Pipeline

This notebook presents the reproducible quantitative workflow for the dissertation **Mapping the Evolution of UK Cryptoasset Regulation: A Computational Analysis of Parliamentary Discourse Using NLP and Network Analytics, 2020–2025**.

The study asks two research questions:

1. How do the thematic priorities of UK parliamentary discourse concerning cryptoasset regulation evolve between 2020 and 2025?
2. Which parliamentary and institutional actors are most strongly associated with the dominant cryptoasset regulatory themes during this period?


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
DATA = ROOT / 'data' / 'processed' / 'hansard_crypto_2020_2025_final.csv'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIG_DIR = ROOT / 'outputs' / 'figures'

df = pd.read_csv(DATA)
print('Contributions:', len(df))
print('Debates/proceedings:', df['debate_id'].nunique())
print('Observed years:', sorted(df['year'].dropna().astype(int).unique().tolist()))


## 1. Corpus coverage

The final screened corpus contains 339 parliamentary contributions across 28 debates and proceedings. Eligible contributions appear in 2021, 2022, 2023, 2024 and 2025. No 2020 contribution meets the documented screening criteria. The study records 2020 as a screened zero and does not create synthetic observations.


In [ ]:
coverage = (
    df.groupby('year')
      .agg(contributions=('speech_id', 'count'),
           debates=('debate_id', 'nunique'),
           speakers=('speaker', 'nunique'),
           words=('word_count', 'sum'))
      .reset_index()
)
coverage


## 2. Quantitative workflow

The analysis uses TF–IDF for exploratory vocabulary analysis and LDA as the principal topic model. Candidate topic counts from K=4 to K=8 are compared through NPMI coherence, topic diversity, seed stability and perplexity. The analysis selects K=5 because it provides the highest NPMI coherence among the tested models.

Yearly mean topic probabilities answer RQ1. A chi-square test with Cramér's V provides supplementary evidence about the association between year and dominant topic. Actor–theme and institution–theme weights answer RQ2 and represent discursive prominence rather than causal political influence.


In [ ]:
analysis_script = ROOT / 'scripts' / 'run_final_analysis.py'
subprocess.run([sys.executable, str(analysis_script)], cwd=ROOT, check=True)


## 3. Model diagnostics

The diagnostic table reports the comparison across candidate topic counts. The selected model uses five topics.


In [ ]:
model_selection = pd.read_csv(TABLE_DIR / 'analysis_model_selection.csv')
model_selection


## 4. Temporal topic prevalence

The yearly table reports mean document–topic probabilities. It shows how the relative prominence of the five regulatory themes changes across the observed years.


In [ ]:
topic_prevalence = pd.read_csv(TABLE_DIR / 'analysis_topic_prevalence_by_year.csv')
topic_prevalence


## 5. Statistical test

The chi-square test examines the relationship between year and dominant topic. The study treats this test as supplementary because contributions cluster within debates and some expected cell counts fall below five.


In [ ]:
chi_square = pd.read_csv(TABLE_DIR / 'analysis_chi_square.csv')
chi_square


## 6. Actor–theme and institution–theme relationships

The actor table uses cumulative and mean LDA topic probabilities for parliamentary speakers. The institution table measures the topic profiles of contributions that mention selected regulatory institutions. These measures indicate discursive association and do not represent causal influence.


In [ ]:
actor_topics = pd.read_csv(TABLE_DIR / 'analysis_actor_topic_weights.csv')
institution_topics = pd.read_csv(TABLE_DIR / 'analysis_institution_topic_weights.csv')
display(actor_topics.head(15))
display(institution_topics)


## 7. Reproducibility and interpretation

The repository stores the final screened corpus, the analysis script, the notebook, model diagnostics, topic probabilities, statistical outputs, actor and institution tables, figures and documentation. Source URLs and Hansard identifiers preserve provenance. The analysis interprets the results as patterns within the screened parliamentary corpus and does not claim that topic prevalence or network weight proves causal political influence.
